# Basic Geometry Operations

Geoprocessing is a set of tools used to analyse and transform geospatial data. These tools allow you to:

- perform spatial operations;
- analyse the spatial relationships between features;
- create new datasets from existing ones.

In this section, we will cover three of the most commonly used geometry operations:

- **Buffer** — creating zones around features at a specified distance;
- **Dissolve** — merging geometries based on a shared attribute;
- **Clip** — cutting one dataset to the extent of another.

## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import osmnx as ox

# keep all downloaded OSM responses in one cache folder at the root of the repository
ox.settings.cache_folder = "../../cache"


### 0.2. Preparing the Data


We start with the boundary of the Tsentralny District of Saint Petersburg.


In [ ]:
area_name = "Tsentralny District, Saint Petersburg"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

We also need bus stops and metro station entrances.


In [ ]:
tags = {
    "highway": "bus_stop",
    "railway": "subway_entrance"
}

stops = ox.features_from_place(area_name, tags)

stops.explore(tiles="cartodbpositron")

The structure of the data:

In [ ]:
stops.head()

As in the [first module](../module_1/spData_4.ipynb), the OSM identifier is stored in the second level of the index rather than in a column, so we pull it out into `osm_id` and keep only the columns we need: the identifier, the two tag columns that tell bus stops and metro entrances apart, the name, and the geometry.

In [ ]:
stops["osm_id"] = stops.index.get_level_values("id")
stops = stops[["osm_id", "railway", "name", "highway", "geometry"]]
stops.head()

The data is now ready for spatial analysis.


## 1. Buffer

A buffer is a zone created around geometric features at a specified distance.

This operation produces catchment areas around features that can be used for further spatial analysis — for example, to identify areas within a given radius or to find features located nearby.

Buffers are created using the `.buffer()` method, where the distance is specified in the units of the coordinate reference system.

If the data is in a **geographic coordinate system** (latitude/longitude), the buffer distance will be interpreted as **degrees**, not metres. So check the CRS first, and reproject if you need to.

> Buffers can only be used correctly in a **projected coordinate system**, where distances are expressed in metres.


Let's create 500-metre buffer zones around the public transport stops.

Before creating the buffers, we need to make sure the data is in a coordinate system that uses metric units.


### 1.1. Checking the CRS and Reprojecting


Check the CRS of the source data.


In [ ]:
stops.crs

The data is in a geographic coordinate system, which is not suitable for our analysis. Let's reproject it into the appropriate UTM zone.


In [ ]:
utm_crs = stops.estimate_utm_crs()

stops_utm = stops.to_crs(utm_crs)

And the CRS after reprojection:


In [ ]:
stops_utm.crs

The CRS now uses metres, so we can proceed with creating the buffer zones.


### 1.2. Creating the Buffers


In [ ]:
# create a copy of the stops GeoDataFrame
stops_buffer = stops_utm.copy()

# build a 500-metre buffer around each stop
stops_buffer = stops_buffer.set_geometry(stops_buffer.geometry.buffer(500))

# inspect the result
stops_buffer.explore(tiles="cartodbpositron")

Once the buffers are created, individual zones may overlap. To produce a single unified catchment area, we need to merge them.


## 2. Dissolve

Dissolve merges multiple geometric features into one, either based on a shared attribute or unconditionally. In GeoPandas, this operation is performed using the `.dissolve()` method.

Dissolve is what you reach for when you need to:

- aggregate features into larger territorial units;
- group data by a shared attribute (e.g. by district);
- produce a single geometry from a collection of individual features.


Let's create a single 500-metre catchment area covering all public transport stops.


When `.dissolve()` is called without specifying a column, all geometries are merged into a single feature.


In [ ]:
stops_dissolved_all = stops_buffer.dissolve()

stops_dissolved_all.explore(tiles="cartodbpositron")

To dissolve by a specific attribute — for example, by stop type — you need to pass the relevant column name.

In our case, let's create two separate catchment areas: one for bus stops and one for metro station entrances. To do this, we need to distinguish features by transport type.

The `railway` column in the source data contains `subway_entrance` for metro entrances and `NaN` for all other features. However, this is not the most convenient field for grouping, so let's create a dedicated transport type column.


In [ ]:
stops_buffer["transport_type"] = "bus"

stops_buffer.loc[
    stops_buffer["railway"] == "subway_entrance",
    "transport_type"
] = "subway"

Now we can dissolve by the new column:


In [ ]:
stops_dissolved_by_type = stops_buffer.dissolve(by="transport_type")
stops_dissolved_by_type.explore(tiles="cartodbpositron")

The result is two separate catchment areas — one for metro station entrances and one for bus stops.


## 3. Clip

Clip extracts the parts of features that fall within a specified boundary. In effect, it trims features to the shape of another layer, keeping only the portions that lie inside it.

This operation is especially useful when you need to constrain the results of an analysis to a particular area.

In our case, after building and dissolving the buffers, we want to keep only the portions of the catchment areas that fall within the district boundary.

Before clipping, always make sure all datasets share the same CRS.


Let's clip the catchment areas to the boundary of the Tsentralny District.


First, let's check whether the district boundary and the dissolved stops share the same CRS, and reproject if necessary.


In [ ]:
admin_border.crs == stops_dissolved_by_type.crs

The CRS values do not match. Let's reproject the district boundary into the same CRS as the stops layer.


In [ ]:
admin_border_utm = admin_border.to_crs(stops_dissolved_by_type.crs)

Now let's clip the catchment areas and inspect the result.


In [ ]:
stops_clipped = stops_dissolved_by_type.clip(admin_border_utm)

stops_clipped.explore(tiles="cartodbpositron")

The result is a set of catchment areas clipped to the boundary of the study area.


## Summary


In this section, we explored three fundamental geometry operations and applied them to analyse public transport accessibility.

We learned:

- how to create buffer zones using `.buffer()`;
- how to merge geometries by attribute using `.dissolve()`;
- how to clip data to a boundary using `.clip()`.